# သင်ခန်းစာ 10 - ထုတ်လုပ်ရေးရှိ AI အေးဂျင့်များ

ဤသင်ခန်းစာတွင် သင်သည် Microsoft Agent Framework (Python) ကို အသုံးပြု၍ AI အေးဂျင့်များအတွက် **ထုတ်လုပ်ရေး ပုံစံများ** ကို လေ့လာပါမည်။ ကျွန်တော်တို့ ဖော်ပြမည့် အရာများမှာ:

- **Observability** — အေးဂျင့်၏ အပြန်အလှန်ဆက်ဆံမှုများတွင် အချိန်တိုင်းတာခြင်းနှင့် မှတ်တမ်းတင်ခြင်း ထည့်သွင်းခြင်း
- **Evaluation** — တုံ့ပြန်မှု အရည်အသွေးကို အဆင့်သတ်မှတ်ရန် အကဲဖြတ် အေးဂျင့်ကို အသုံးပြုခြင်း
- **Cost management** — token ကို ထိရောက်စွာ အသုံးပြုခြင်းနှင့် မော်ဒယ် ရွေးချယ်မှုတို့အတွက် မဟာဗျူဟာများ

အဆိုပါ အခြေအနေမှာ အသုံးပြုသူများ၏ ခရီးစဉ်အစီအစဉ်ရေးဆွဲရာတွင် အကူအညီပေးသော **ခရီးသွား အေးဂျင့်** ဖြစ်ပြီး အပေါ်ယံတွင် စောင့်ကြည့်မှုနှင့် အကဲဖြတ်မှု အလွှာများကို ထပ်ဆင့် ထည့်သွင်းထားသည်။


## ဆက်တင်


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ထုတ်လုပ်ရေးတွင်စဉ်းစားရမည့်အချက်များ

AI အေးဂျင်များကို နမူနာများမှ ထုတ်လုပ်ရေးသို့ ရွှေ့ရာတွင် အောက်ပါ သုံးခုအပေါ် အထူးဂရုပြုရပါသည်:

1. **Observability** — အေးဂျင်က ဘာလုပ်နေလဲ၊ ဘာလောက်ကြာနေရလဲ၊ ဘယ်ကိရိယာများကို ခေါ်သုံးနေလဲ စရှိသော အချက်များကို မြင်သာစေလို့ လိုအပ်သည်။ tracing နှင့် logging မရှိပါက ထုတ်လုပ်ရေးမှ ဖြစ်ပေါ်သော ပြဿနာများကို debug လုပ်၍ ဖြေရှင်းရန် အလွန်ခက်ခဲပါသည်။

2. **Evaluation** — အလိုအလျောက် အရည်အသွေး စစ်ဆေးမှုများက အေးဂျင်၏ တုံ့ပြန်ချက်များကို အချိန်အတွင်း တိကျ၊ ပြည့်စုံနှင့် အသုံးဝင်နေစေသည်။ အကဲဖြတ်သူ အေးဂျင်တစ်ခုက သတ်မှတ်ထားသော ချက်စည်းများနှင့် နှိုင်းယှဉ်ပြီး တုံ့ပြန်ချက်များကို အမှတ်ပေးနိုင်သည်။

3. **Cost Management** — Token အသုံးပြုမှုသည် တိုက်ရိုက် စရိတ်ကို ထိခိုက်စေသည်။ prompt ကို အထိရောက်ဆုံး ပြုပြင်ခြင်း၊ မော်ဒယ်ရွေးချယ်ခြင်း၊ caching စသည့် မဟာဗျူဟာများက အရည်အသွေးမလျော့ချဘဲ စရိတ်ကို ထိန်းချုပ်ထားရာတွင် ကူညီပေးနိုင်သည်။


## ကြည့်ရှုနိုင်သော အေးဂျင့် တည်ဆောက်ခြင်း

ကျွန်ုပ်တို့သည် ခရီးသွားကိရိယာများကို သတ်မှတ်ပြီး အေးဂျင့်ခေါ်ဆိုမှုကို အချိန်တိုင်းတာမှုဖြင့် ဝိုက်ပတ်ထားသဖြင့် နှောင့်နှေးချိန်ကို စောင့်ကြည့်နိုင်ပါသည်။ ထုတ်လုပ်ရေးပတ်ဝန်းကျင်တွင် သင်သည် OpenTelemetry သို့မဟုတ် ဆင်တူသော tracing backend တစ်ခုနှင့် ပေါင်းစည်းနိုင်ပါသည်။


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## အကဲဖြတ်ပုံစံများ

ထုတ်လုပ်မှုတွင် အထွေထွေသုံးလေ့ရှိသော ပုံစံတစ်ခုမှာ ဒုတိယ အေးဂျင့်ကို **အကဲဖြတ်သူ** အဖြစ် အသုံးပြုခြင်း ဖြစ်သည်။ အကဲဖြတ်သူသည် အဓိက အေးဂျင့်၏ တုံ့ပြန်ချက်ကို ပြီးပြည့်မှု၊ တိကျမှု နှင့် အကူအညီပေးနိုင်မှု ကဲ့သို့ ရှေ့တင် သတ်မှတ်ထားသော စံချိန်များနှင့် တိုက်ရိုက် နှိုင်းယှဥ်ကာ အဆင့်သတ်မှတ်သည်။

ဒါက အောက်ပါအရာများကို ခွင့်ပြုသည်:
- အသုံးပြုသူထံ ရောက်ရှိမီ တုံ့ပြန်ချက်များအတွက် အလိုအလျောက် အရည်အသွေး စစ်ဆေးချက်များ
- prompt များ သို့မဟုတ် မော်ဒယ်များ ပြောင်းလဲသည့်အခါ ပြန်လည်ကျဆင်းမှု ရှာဖွေရေး
- အေးဂျင့်၏ ဆောင်ရွက်မှုပုံစံကို အချိန်နှင့်အမျှ ဆက်တိုက် ကြည့်ကြပ်ခြင်း


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု မဟာဗျူဟာများ

ကုန်ကျစရိတ်ကို ထိန်းချုပ်ခြင်းသည် ထုတ်လုပ်ရေးအဆင့်ရှိ AI အေးဂျင့်များအတွက် အရေးကြီးသည်။ အဓိက မဟာဗျူဟာများမှာ အောက်ပါအတိုင်းဖြစ်သည်။

| နည်းဗျူဟာ | ဖော်ပြချက် |
|---|---|
| **Prompt တိုးတက်အောင် ပြုပြင်ခြင်း** | စနစ် အညွှန်းများကို တိုတောင်းစွာထားပါ။ ထပ်တလဲလဲသော အကြောင်းအရာများကို ဖယ်ရှား၍ input token များ လျော့နည်းစေပါ။ |
| **မော်ဒယ် ရွေးချယ်မှု** | ခွဲခြားခြင်း (classification) သို့မဟုတ် ထုတ်ယူခြင်း (extraction) ကဲ့သို့ ရိုးရှင်းသော တာဝန်များအတွက် ပိုမိုသေးငယ်ပြီး စျေးကြမ်းတဲ့ မော်ဒယ်များကို (ဥပမာ GPT-4o-mini) အသုံးပြုပါ၊ ရှုပ်ထွေးသော အတွေးခေါ်ချက်များအတွက်တော့ စွမ်းရည်ကောင်းသော မော်ဒယ်ကြီးများကို သီးသန့်ထားပါ။ |
| **ကက်ရှ် အား အသုံးပြုခြင်း** | ကိရိယာရလဒ်များနှင့် မကြာခဏ မေးမြန်းမှုများကို ကက်ရှ်ထဲသိမ်းဆည်းထား၍ ထပ်စဉ် API ခေါ်ဆိုမှုများကို တားဆီးပါ။ |
| **Token ဘတ်ဂျက်များ** | မမျှော်လင့်ရသော အကြာကြီး ဖြေကြားချက်များ မထွက်ဖို့ `max_tokens` ကန့်သတ်ချက်များ သတ်မှတ်ပါ။ |
| **အစုစည်းလုပ်ခြင်း (Batching)** | ပြုလုပ်နိုင်လျှင် အသုံးပြုသူ မေးခွန်းများ အများအပြားကို တစ်ခုတည်းသော API ခေါ်ဆိုမှုတစ်ခုအဖြစ် အုပ်စုဖွဲ့ပါ။ |

လက်တွေ့တွင် အဆင့်ခွဲသော နည်းလမ်းတစ်ခု ကောင်းမွန်သည် — ရိုးရှင်းသော တောင်းဆိုချက်များကို လျင်မြန်ပြီး စျေးသက်သာသော မော်ဒယ်တစ်ခုသို့ ညွှန်ပေး၍ ရှုပ်ထွေးသော မေးခွန်းများကိုသာ စွမ်းရည်ကောင်းသော မော်ဒယ်သို့ တင်မောင်းပါ။


## အကျဥ်းချုပ်

ဤသင်ခန်းစာတွင် ဤအရာများကို သင် သင်ယူထားပါသည်:

1. **Add observability** ကို အေဂျင့် ဆက်သွယ်မှုများတွင် အချိန်တိုင်းတာခြင်းနှင့် မှတ်တမ်းတင်ခြင်းဖြင့် ထည့်သွင်း၍ tracing နှင့် monitoring အတွက် အခြေခံကို တည်ဆောက်သည်။
2. **Evaluate agent responses** ကို အကဲဖြတ်အေဂျင့် တစ်ခု အသုံးပြု၍ အလိုအလျောက် အဖြေများကို ဆန်းစစ်ကာ ပြည့်စုံမှု၊ တိကျမှုနှင့် အထောက်အကူရှိမှုတို့ကို အဆင့်သတ်မှတ်သည်။
3. **Manage costs** ကို prompt optimization, model selection, caching, နှင့် token budgets များအားဖြင့် စီမံခန့်ခွဲသည်။

ဤထုတ်လုပ်မှု ပုံစံများသည် သင့် AI အေဂျင့်များကို ယုံကြည်စိတ်ချရ၊ တိုင်းတာနိုင်ပြီး၊ အရွယ်အစားကြီးလာသည့်အခါတွင် ကုန်ကျစရိတ်ထိရောက်စေရန် ကူညီပေးသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
အသိပေးချက်:
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု Co‑op Translator (https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်ရေးအတွက် ကြိုးပမ်းသော်လည်း အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှားယွင်းမှုများ ရှိနိုင်ကြောင်း ကျေးဇူးပြု၍ သတိပြုပါ။ မူရင်းစာတမ်းကို ကိုးကားရသော အရင်းအမြစ်အဖြစ် သတ်မှတ်ထားသင့်ပြီး အရေးကြီးသော အချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူဘာသာပြန်ဖြင့် အတည်ပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုမှုကြောင့် ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မှားယွင်းဖော်ပြချက်များအတွက် ကျွန်ုပ်တို့သည် တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
